<a href="https://colab.research.google.com/github/AravindNRaj/TP53-Functionality-/blob/main/Tp53_functionality_project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [9]:
import pandas as pd
import numpy as np
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import roc_auc_score, accuracy_score, precision_score, recall_score, f1_score, matthews_corrcoef, make_scorer
from sklearn.model_selection import StratifiedKFold, cross_validate

# --- Configuration (re-using from previous context) ---
FILE_PATH = '/content/TumorVariantDownload_r21 (TP533 cancer data).csv'
RANDOM_SEED = 42
DEV_BOUNDARY = 15000 # User-specified training data boundary
TARGET = 'TransactivationClass'
FEATURE = 'SIFTClass' # User-specified sole feature

# Set random seed for reproducibility
np.random.seed(RANDOM_SEED)

# --- 1. Data Loading & Strict Row Split ---
# Using raw_df, dev_df, holdout_df already loaded in previous cells for consistency
raw_df = pd.read_csv(FILE_PATH)
raw_df['_original_row_number'] = np.arange(1, len(raw_df) + 1)

dev_df = raw_df.iloc[:DEV_BOUNDARY].copy()
holdout_df = raw_df.iloc[DEV_BOUNDARY:].copy()

print(f"Development set size: {len(dev_df)} rows")
print(f"Holdout set size: {len(holdout_df)} rows")

# --- 2. Target Mapping ---
def binary_mapper(val):
    s = str(val).lower().strip()
    if s in ['functional', 'supertrans']: return 1
    if s in ['non-functional', 'partially functional']: return 0
    return np.nan

dev_df['target_binary'] = dev_df[TARGET].apply(binary_mapper)
holdout_df['target_binary'] = holdout_df[TARGET].apply(binary_mapper)

# Filter out rows with NaN targets for model training/evaluation
dev_model_df = dev_df.dropna(subset=['target_binary']).copy()
holdout_eval_df = holdout_df.dropna(subset=['target_binary']).copy()

print(f"Development set with valid targets for modeling: {len(dev_model_df)} rows")
print(f"Holdout set with valid targets for evaluation: {len(holdout_eval_df)} rows")

# --- 3. Prepare Feature and Target for Model ---
X_train = dev_model_df[[FEATURE]]
y_train = dev_model_df['target_binary']

X_test = holdout_df[[FEATURE]] # Use original holdout for prediction, then filter for evaluation
y_test_eval = holdout_eval_df['target_binary']

# SIFTClass is typically a categorical feature (e.g., 'deleterious', 'tolerated')
numerical_features = []
categorical_features = [FEATURE]

# --- 4. Build Machine Learning Pipeline ---
# Preprocessing for categorical features (SIFTClass)
categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='constant', fill_value='missing')), # Handle missing SIFTClass values
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False)) # One-hot encode SIFTClass
])

# Create a preprocessor using ColumnTransformer
preprocessor = ColumnTransformer(
    transformers=[
        ('cat', categorical_transformer, categorical_features)
    ],
    remainder='drop' # Drop any other columns not explicitly specified
)

# Define the model (HistGradientBoostingClassifier as used previously in the notebook)
model = HistGradientBoostingClassifier(random_state=RANDOM_SEED)

# Create the full pipeline
sift_model_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', model)
])

# --- 5. Cross-Validation on Development Set ---
print(f"\nPerforming 5-fold cross-validation on the development set using {FEATURE}...")

# Define custom scorer for MCC since it's not directly available in `scoring` dictionary with cross_validate
mcc_scorer = make_scorer(matthews_corrcoef)

scoring = {
    'roc_auc': 'roc_auc',
    'accuracy': 'accuracy',
    'precision': 'precision',
    'recall': 'recall',
    'f1': 'f1',
    'mcc': mcc_scorer
}

# Stratified K-Fold for classification tasks
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_SEED)
cv_results = cross_validate(sift_model_pipeline, X_train, y_train, cv=skf, scoring=scoring, return_train_score=False)

print("\n--- Cross-Validation Results ---")
for metric_name, scores in cv_results.items():
    if metric_name.startswith('test_'):
        print(f"{metric_name[5:].replace('_', ' ').title()} (Mean): {np.mean(scores):.4f} (Std: {np.std(scores):.4f})")

# --- 6. Train the Final Model on the Entire Development Set ---
print(f"\nTraining the final model on the entire development set using {FEATURE}...")
sift_model_pipeline.fit(X_train, y_train)
print("Final model training complete.")

# --- 7. Evaluate on Locked Holdout Set ---
print("\nEvaluating final model on the holdout set...")
# Predict probabilities for ROC AUC
holdout_pred_probs = sift_model_pipeline.predict_proba(X_test)[:, 1]
# Predict classes for other metrics
holdout_pred_classes = sift_model_pipeline.predict(X_test)

# Add predictions to the holdout_df for easier handling and debugging
holdout_df['sift_pred_prob'] = holdout_pred_probs
holdout_df['sift_pred_class'] = holdout_pred_classes

# Filter for rows with valid targets for evaluation
holdout_eval_df_with_preds = holdout_df.dropna(subset=['target_binary']).copy()

# Recalculate predictions for eval_df (using predicted values from the full holdout_df, matching original indexes)
eval_pred_probs = holdout_eval_df_with_preds['sift_pred_prob']
eval_pred_classes = holdout_eval_df_with_preds['sift_pred_class']

# Calculate evaluation metrics
roc_auc = roc_auc_score(y_test_eval, eval_pred_probs)
accuracy = accuracy_score(y_test_eval, eval_pred_classes)
precision = precision_score(y_test_eval, eval_pred_classes, zero_division=0)
recall = recall_score(y_test_eval, eval_pred_classes, zero_division=0)
f1 = f1_score(y_test_eval, eval_pred_classes, zero_division=0)
mcc = matthews_corrcoef(y_test_eval, eval_pred_classes)

print(f"\n--- Holdout Evaluation Metrics (using {FEATURE}) ---")
print(f"ROC AUC: {roc_auc:.4f}")
print(f"Accuracy: {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall: {recall:.4f}")
print(f"F1-Score: {f1:.4f}")
print(f"Matthews Correlation Coefficient (MCC): {mcc:.4f}")

Development set size: 15000 rows
Holdout set size: 14891 rows
Development set with valid targets for modeling: 10980 rows
Holdout set with valid targets for evaluation: 10628 rows

Performing 5-fold cross-validation on the development set using SIFTClass...

--- Cross-Validation Results ---
Roc Auc (Mean): 0.7390 (Std: 0.0188)
Accuracy (Mean): 0.9312 (Std: 0.0057)
Precision (Mean): 0.6296 (Std: 0.0403)
Recall (Mean): 0.5063 (Std: 0.0347)
F1 (Mean): 0.5612 (Std: 0.0372)
Mcc (Mean): 0.5281 (Std: 0.0403)

Training the final model on the entire development set using SIFTClass...
Final model training complete.

Evaluating final model on the holdout set...

--- Holdout Evaluation Metrics (using SIFTClass) ---
ROC AUC: 0.7454
Accuracy: 0.9238
Precision: 0.6256
Recall: 0.5244
F1-Score: 0.5705
Matthews Correlation Coefficient (MCC): 0.5315
